# Managing Team State - Saving and Loading Teams

In [1]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "gpt-4o",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)

In [2]:
from autogen_agentchat.conditions import MaxMessageTermination

agent_1 = AssistantAgent(
    name='Writer_1',
    model_client=model_client,
    system_message="You are a helpful assistant. Give the output in less than 30 words",
)

agent_2 = AssistantAgent(
    name='Writer_2',
    model_client=model_client,
    system_message="You are a helpful assistant.Give the output in less than 30 words",
)

terminationCondition = MaxMessageTermination(max_messages=3)

agent_team = RoundRobinGroupChat(participants=[agent_1, agent_2],termination_condition=terminationCondition)

In [8]:
from autogen_agentchat.ui import Console

stream = agent_team.run_stream(task="Write a poem about the sea in 3 lines")

await Console(stream)

---------- TextMessage (user) ----------
Write a poem about the sea in 3 lines
---------- TextMessage (Writer_1) ----------
Morning mist curls over waves, whispering secrets,  
Salt air stirs memories of distant horizons,  
Ocean’s heartbeat drifts beneath the sky.
---------- TextMessage (Writer_2) ----------
Salt spray kisses dawn,  
Waves whisper old lullabies,  
Stars cradle the horizon.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a poem about the sea in 3 lines', type='TextMessage'), TextMessage(source='Writer_1', models_usage=RequestUsage(prompt_tokens=237, completion_tokens=214), metadata={}, content='Morning mist curls over waves, whispering secrets,  \nSalt air stirs memories of distant horizons,  \nOcean’s heartbeat drifts beneath the sky.', type='TextMessage'), TextMessage(source='Writer_2', models_usage=RequestUsage(prompt_tokens=275, completion_tokens=306), metadata={}, content='Salt spray kisses dawn,  \nWaves whisper old lullabies,  \nStars cradle the horizon.', type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')

In [10]:
team_state = await agent_team.save_state()
print("Team state saved.")
print(team_state)

Team state saved.
{'type': 'TeamState', 'version': '1.0.0', 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState', 'version': '1.0.0', 'agent_state': {'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'What was the last line of Poem you wrote?', 'source': 'user', 'type': 'UserMessage'}, {'content': "I haven't written a poem in this chat yet. If you'd like, I can write one now!", 'thought': None, 'source': 'Writer_1', 'type': 'AssistantMessage'}, {'content': "I haven't written a poem in this chat yet. I can write one now if you'd like!", 'source': 'Writer_2', 'type': 'UserMessage'}, {'content': 'What was the last line of Poem you wrote?', 'source': 'user', 'type': 'UserMessage'}, {'content': 'I haven’t written a poem in this chat, so there’s no last line to recall.', 'thought': None, 'source': 'Writer_1', 'type': 'AssistantMessage'}, {'content': 'I haven’t written a poem in this chat, so there’s no last line to recall.', 'source': 'Wri

In [5]:
await agent_team.reset()

In [11]:

stream = agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- TextMessage (user) ----------
What was the last line of Poem you wrote?
---------- TextMessage (Writer_1) ----------
Ocean’s heartbeat drifts beneath the sky.
---------- TextMessage (Writer_2) ----------
Ocean’s heartbeat drifts beneath the sky.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(source='Writer_1', models_usage=RequestUsage(prompt_tokens=316, completion_tokens=100), metadata={}, content='Ocean’s heartbeat drifts beneath the sky.', type='TextMessage'), TextMessage(source='Writer_2', models_usage=RequestUsage(prompt_tokens=333, completion_tokens=103), metadata={}, content='Ocean’s heartbeat drifts beneath the sky.', type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')

In [7]:
await agent_team.load_state(team_state)


In [8]:

stream = agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- user ----------
What was the last line of Poem you wrote?
---------- Writer_1 ----------
Secrets in the deep.
---------- Writer_2 ----------
Secrets in the deep.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(source='Writer_1', models_usage=RequestUsage(prompt_tokens=111, completion_tokens=6), metadata={}, content='Secrets in the deep.', type='TextMessage'), TextMessage(source='Writer_2', models_usage=RequestUsage(prompt_tokens=124, completion_tokens=6), metadata={}, content='Secrets in the deep.', type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')

In [12]:
team_state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'What was the last line of Poem you wrote?',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': "I haven't written a poem in this chat yet. If you'd like, I can write one now!",
       'thought': None,
       'source': 'Writer_1',
       'type': 'AssistantMessage'},
      {'content': "I haven't written a poem in this chat yet. I can write one now if you'd like!",
       'source': 'Writer_2',
       'type': 'UserMessage'},
      {'content': 'What was the last line of Poem you wrote?',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': 'I haven’t written a poem in this chat, so there’s no last line to recall.',
       'thought': None,
       'source': 'Writer_1',
       'type': 'AssistantMessage'

In [13]:
type(team_state)

dict

# if its a dict, it can be serialized to a file or written to a database

In [14]:
import json

In [15]:
with open('team_state.json', 'w') as f:
    json.dump(team_state, f)